# Desafio 1 - GMM para clusterização do Iris

Agrupamento das flores do Iris com Gaussian Mixture Model, com os hiperparâmetros otimizados pelo Optuna (menor BIC).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score

In [ ]:
# Importar o dataset
iris = load_iris()
X = iris.data
y = iris.target

In [ ]:
# Exploração
df = pd.DataFrame(X, columns=iris.feature_names)
print(df.shape)
print(df.dtypes)
df.describe()

In [ ]:
# Pré-processamento
X_scaled = StandardScaler().fit_transform(X)

In [ ]:
# Otimização com Optuna
def objective(trial):
    gmm = GaussianMixture(
        n_components=trial.suggest_int('n_components', 2, 10),
        covariance_type=trial.suggest_categorical('covariance_type', ['full', 'tied', 'diag', 'spherical']),
        random_state=42,
    )
    gmm.fit(X_scaled)
    return gmm.bic(X_scaled)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)
study.best_params

In [ ]:
# Treinar com os melhores hiperparâmetros e clusterizar
gmm = GaussianMixture(**study.best_params, random_state=42)
clusters = gmm.fit_predict(X_scaled)

In [ ]:
# Avaliação comparando com as classes reais
print('Adjusted Rand Index:', adjusted_rand_score(y, clusters))
pd.crosstab(y, clusters, rownames=['classe real'], colnames=['cluster'])

In [ ]:
# Visualização
plt.scatter(df['petal length (cm)'], df['petal width (cm)'], c=clusters)
plt.xlabel('petal length (cm)')
plt.ylabel('petal width (cm)')
plt.show()

## Resultados

O Optuna escolheu 2 componentes com covariância full, que teve o menor BIC. Com isso, a setosa ficou isolada em um cluster e versicolor e virginica ficaram juntas no outro, já que as duas espécies se sobrepõem bastante. O Adjusted Rand Index foi de 0,57.